In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.utils import resample
from collections import defaultdict

In [2]:
# TCVファイルの読み込み
train_data = pd.read_csv('../000.data/train/train_A.tsv', sep="\t")
test = pd.read_csv('../000.data/test.tsv', sep="\t")

In [3]:
# 末尾が "_A" のテストデータだけを抽出
test_data = test[test["user_id"].str.endswith("_A")]

In [4]:
# 関連度スコアの設定
def compute_relevance(row):
    if row["event_type"] == 3 and row["ad"] == 1:  # 広告経由の購入
        return 3
    elif row["event_type"] == 2:  # 広告クリック
        return 2
    elif row["event_type"] == 1:  # 詳細ページ閲覧
        return 1
    else:  # カート追加
        return 0

train_data["relevance"] = train_data.apply(compute_relevance, axis=1)

In [5]:
# タイムスタンプをdatetime型に変換
train_data["time_stamp"] = pd.to_datetime(train_data["time_stamp"])

In [6]:
# 最新の行動からの経過時間（秒）
train_data["time_since_last"] = (train_data["time_stamp"].max() - train_data["time_stamp"]).dt.total_seconds()

In [7]:
# ユーザーごとの行動回数集計
user_features = train_data.groupby("user_id").agg(
    user_total_interactions=("product_id", "count"),
    user_unique_products=("product_id", "nunique"),
    user_avg_time_since=("time_since_last", "mean")
).reset_index()

In [8]:
# 商品ごとの人気度
product_features = train_data.groupby("product_id").agg(
    product_total_interactions=("user_id", "count"),
    product_unique_users=("user_id", "nunique")
).reset_index()

In [9]:
# データ結合
train_data = train_data.merge(user_features, on="user_id", how="left")
train_data = train_data.merge(product_features, on="product_id", how="left")

In [10]:
# 学習用特徴量
features = [
    "user_total_interactions", "user_unique_products", "user_avg_time_since",
    "product_total_interactions", "product_unique_users"
]

In [11]:
# クラス不均衡に対する学習データのバランス調整
train_data_pos = train_data[train_data["relevance"] >= 2]  # 広告クリックと購入のデータ
train_data_neg = train_data[train_data["relevance"] < 2]   # その他のデータ

In [12]:
# 多数派クラス（詳細閲覧・カート）をアンダーサンプリング
train_data_neg_undersampled = resample(train_data_neg, 
                                      replace=True, 
                                      n_samples=500000,
                                      random_state=42)

In [13]:
# 少数派クラス（購入・広告クリック）をオーバーサンプリング
train_data_pos_oversampled = resample(train_data_pos, 
                                      replace=True, 
                                      n_samples=1000000,
                                      random_state=42)

In [14]:
# サンプリング後のデータ
train_data_balanced = pd.concat([train_data_neg_undersampled, train_data_pos_oversampled])

In [15]:
# クロスバリデーション設定
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [16]:
# 評価指標を格納するリスト
ndcg_scores = []

In [17]:
for train_index, val_index in kf.split(train_data_balanced):
    train_set = train_data_balanced.iloc[train_index]
    val_set = train_data_balanced.iloc[val_index]
    
    # 特徴量とラベル
    X_train = train_set[features]
    y_train = train_set["relevance"]
    X_val = val_set[features]
    y_val = val_set["relevance"]
    
    # クエリごとのデータ数（ユーザーごと）
    group_train = train_set.groupby("user_id").size().to_numpy()
    group_val = val_set.groupby("user_id").size().to_numpy()
    
    # XGBoost DMatrix に変換
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtrain.set_group(group_train)

    dval = xgb.DMatrix(X_val, label=y_val)
    dval.set_group(group_val)

    # 学習パラメータ設定（過学習防止のための調整）
    params = {
        "objective": "rank:ndcg",
        "eval_metric": "ndcg",
        "booster": "gbtree",
        "eta": 0.1,
        "max_depth": 6,
        "min_child_weight": 10,
        "subsample": 0.8,               # サブサンプリング（学習データの80%を使用）
        "colsample_bytree": 0.8,        # 特徴量のサブサンプリング
        "lambda": 1.0,                  # L2正則化
        "gamma": 0.1,                   # 木の分岐時の正則化
    }
    
    # モデル学習
    evals_result = {}  # 追加: early stoppingのログを格納
    model = xgb.train(
        params, dtrain, num_boost_round=200,
        evals=[(dtrain, "train"), (dval, "val")],  # 検証データを明示
        evals_result=evals_result,  # 追加: early stoppingのための評価ログ
        early_stopping_rounds=20,  # early stopping を適用
        verbose_eval=10
    )
    
    # バリデーションデータの nDCG 計算
    ground_truth_val = defaultdict(dict)
    for _, row in val_set.iterrows():
        ground_truth_val[row["user_id"]][row["product_id"]] = row["relevance"]

    # バリデーションデータのコピーを作成
    val_set_copy = val_set.copy()

    # バリデーションデータに対してスコアを設定
    val_set_copy["score"] = model.predict(xgb.DMatrix(val_set_copy[features]))

    # nDCG の計算
    def dcg_at_k(relevance_scores, k):
        relevance_scores = np.asarray(relevance_scores)[:k]
        return np.sum(relevance_scores / np.log2(np.arange(2, relevance_scores.size + 2)))

    def ndcg_at_k(relevance_scores, k):
        dcg = dcg_at_k(relevance_scores, k)
        ideal_dcg = dcg_at_k(sorted(relevance_scores, reverse=True), k)
        return dcg / ideal_dcg if ideal_dcg > 0 else 0

    def evaluate_ndcg(data, ground_truth, k=22):
        ndcg_scores = []
        for user_id, group in data.groupby("user_id"):
            predicted_items = group.sort_values("score", ascending=False)["product_id"].tolist()
            relevance_scores = [ground_truth[user_id].get(item, 0) for item in predicted_items]
            ndcg_scores.append(ndcg_at_k(relevance_scores, k))
        return np.mean(ndcg_scores) if ndcg_scores else 0
    
    ndcg_val = evaluate_ndcg(val_set_copy, ground_truth_val, k=22)
    ndcg_scores.append(ndcg_val)
    print(f"Validation nDCG@22: {ndcg_val:.4f}")

[0]	train-ndcg:0.93993	val-ndcg:0.95725
[10]	train-ndcg:0.94452	val-ndcg:0.95917
[20]	train-ndcg:0.94620	val-ndcg:0.96025
[30]	train-ndcg:0.94768	val-ndcg:0.96126
[40]	train-ndcg:0.94852	val-ndcg:0.96196
[50]	train-ndcg:0.94937	val-ndcg:0.96252
[60]	train-ndcg:0.94998	val-ndcg:0.96286
[70]	train-ndcg:0.95056	val-ndcg:0.96324
[80]	train-ndcg:0.95135	val-ndcg:0.96376
[90]	train-ndcg:0.95193	val-ndcg:0.96408
[100]	train-ndcg:0.95246	val-ndcg:0.96428
[110]	train-ndcg:0.95288	val-ndcg:0.96464
[120]	train-ndcg:0.95330	val-ndcg:0.96481
[130]	train-ndcg:0.95369	val-ndcg:0.96500
[140]	train-ndcg:0.95417	val-ndcg:0.96521
[150]	train-ndcg:0.95455	val-ndcg:0.96541
[160]	train-ndcg:0.95493	val-ndcg:0.96557
[170]	train-ndcg:0.95525	val-ndcg:0.96570
[180]	train-ndcg:0.95552	val-ndcg:0.96587
[190]	train-ndcg:0.95577	val-ndcg:0.96600
[199]	train-ndcg:0.95599	val-ndcg:0.96618
Validation nDCG@22: 0.8249
[0]	train-ndcg:0.93945	val-ndcg:0.95768
[10]	train-ndcg:0.94429	val-ndcg:0.95966
[20]	train-ndcg:0.946

In [18]:
# クロスバリデーションの平均 nDCG を表示
print(f"Average nDCG@22 from CV: {np.mean(ndcg_scores):.4f}")

Average nDCG@22 from CV: 0.8249


In [19]:
# テストデータの処理
candidate_products = train_data.groupby("product_id")["user_id"].count().reset_index()
candidate_products = candidate_products.sort_values("user_id", ascending=False).head(22)  # 上位22商品

In [20]:
# すべてのユーザーに候補商品を紐づける
test_data = test_data.assign(key=1).merge(candidate_products.assign(key=1), on="key").drop("key", axis=1)
test_data = test_data.rename(columns={'user_id_x': 'user_id'}).drop(columns=['user_id_y'])

In [21]:
# 評価データの前処理（学習データと同じ特徴量処理を適用）
test_data = test_data.merge(user_features, on="user_id", how="left")
test_data = test_data.merge(product_features, on="product_id", how="left")
test_data.fillna(0, inplace=True)

In [22]:
# 予測用データ
X_test = test_data[features]
dtest = xgb.DMatrix(X_test)

In [23]:
# 予測スコアの計算
test_data["score"] = model.predict(dtest)

In [24]:
# 各ユーザーごとにランキング付け（スコアが高い順）
test_data["rank"] = test_data.groupby("user_id")["score"].rank(ascending=False, method="first")

In [25]:
# テストデータの nDCG 計算
ground_truth_test = defaultdict(dict)
for _, row in train_data.iterrows():
    ground_truth_test[row["user_id"]][row["product_id"]] = row["relevance"]

ndcg_test = evaluate_ndcg(test_data, ground_truth_test, k=22)
print(f"Test nDCG@22: {ndcg_test:.4f}")

Test nDCG@22: 0.2619


In [26]:
# 提出用データの作成（上位 22 件のみ）
submission = test_data[test_data["rank"] <= 22][["user_id", "product_id", "rank"]]
submission['rank'] = submission['rank'].astype(int)

In [27]:
# 保存
submission.to_csv('./predict_file/predictions_A.tsv', sep="\t", index=False)